In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Tuple, Dict, Optional
from openpyxl import load_workbook
import os

# OR-Tools for Constraint Programming
from ortools.sat.python import cp_model

In [ ]:
Rooms = pd.read_excel("../../data/Raw Data/Data.xlsx" , sheet_name="Rooms")
Courses = pd.read_excel("../../data/Raw Data/Data.xlsx" , sheet_name="Courses" )
Doctors = pd.read_excel("../../data/Raw Data/Data.xlsx" , sheet_name="Doctors" )
Divisions = pd.read_excel("../../data/Raw Data/Data.xlsx" , sheet_name="Division" )

In [ ]:
Rooms.head(10)

In [ ]:
Courses.head(10)

In [ ]:
Doctors.head(10)

In [ ]:
Divisions.head(20)

In [ ]:
def parse_time(t):
    h, m = map(int, str(t).split(":"))
    return h * 60 + m

def time_ranges_overlap(start1, end1, start2, end2):
    return not (end1 <= start2 or end2 <= start1)

def parse_time_extended(time_str):
    try:
        if isinstance(time_str, str):
            time_str = time_str.strip().upper()
            
            is_pm = 'PM' in time_str
            is_am = 'AM' in time_str
            
            time_str = time_str.replace('AM', '').replace('PM', '').strip()
            
            parts = time_str.split(':')
            hours = int(parts[0])
            minutes = int(parts[1]) if len(parts) > 1 else 0
            
            if hours == 12:
                if is_am:
                    hours = 0  
            elif is_pm and hours != 12:
                hours += 12  
            
            return hours * 60 + minutes
        return 0
    except:
        return 0

def minutes_to_time_str(minutes):
    if minutes == 0:
        return "12:00 AM"
    hours = minutes // 60
    mins = minutes % 60
    hours = hours % 24
   
    if hours == 0:
        hour_12 = 12
        period = "AM"
    elif hours < 12:
        hour_12 = hours
        period = "AM"
    elif hours == 12:
        hour_12 = 12
        period = "PM"
    else:
        hour_12 = hours - 12
        period = "PM"
    
    return f"{hour_12}:{mins:02d} {period}"

def is_time_in_range(time_minutes, start_minutes, end_minutes):
    if end_minutes < start_minutes: 
        return time_minutes >= start_minutes or time_minutes <= end_minutes
    return start_minutes <= time_minutes <= end_minutes

def time_to_minutes(time_value):
    from datetime import time as dt_time
    if isinstance(time_value, (int, float)):
        return int(time_value)

    if isinstance(time_value, dt_time):
        return time_value.hour * 60 + time_value.minute
    
    if isinstance(time_value, str):
        result = parse_time_extended(time_value)
        if result > 0:
            return result
        
        try:
            return parse_time(time_value)
        except:
            return 0
    
    return 0

print("Helper functions defined")

In [ ]:
# Prepare data structures
room_dict = {row['Room_ID']: {'capacity': row['Capacity'], 'type': row['Type']} 
             for _, row in Rooms.iterrows()}

course_dict = {row['Course_ID']: {
    'instructor': row['Instructor_ID'],
    'days': row['Days'],
    'hours_per_day': row['Hours_per_day'],
    'type': row['Type'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Courses.iterrows()}

doctor_availability = {}
for _, row in Doctors.iterrows():
    inst_id = row['Instructor_ID']
    day = row['Day']
    start = row['Start_Time']
    end = row['End_Time']
    start_minutes = time_to_minutes(start)
    end_minutes = time_to_minutes(end)
    
    if inst_id not in doctor_availability:
        doctor_availability[inst_id] = {}
    if day not in doctor_availability[inst_id]:
        doctor_availability[inst_id][day] = []
    doctor_availability[inst_id][day].append((start_minutes, end_minutes))

division_dict = {row['Num_ID']: {
    'students': row['StudentNum'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Divisions.iterrows()}

lecture_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lecture']
lab_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lab']

print(f"Data prepared: {len(course_dict)} courses, {len(room_dict)} rooms, {len(division_dict)} divisions")
print(f"Lecture rooms: {len(lecture_rooms)}, Lab rooms: {len(lab_rooms)}")

In [ ]:
class SchedulingCP:
    """Constraint Programming scheduler using OR-Tools CP-SAT."""
    
    def __init__(self, courses_df, rooms_df, doctors_df, divisions_df, 
                 time_limit_seconds=300, use_optimization=True):
        """Initialize the CP scheduler.
        
        Args:
            courses_df: DataFrame with course information
            rooms_df: DataFrame with room information
            doctors_df: DataFrame with doctor availability
            divisions_df: DataFrame with division information
            time_limit_seconds: Maximum time to search for solution
            use_optimization: Whether to optimize for better solutions
        """
        self.courses_df = courses_df
        self.rooms_df = rooms_df
        self.doctors_df = doctors_df
        self.divisions_df = divisions_df
        self.time_limit_seconds = time_limit_seconds
        self.use_optimization = use_optimization
        
        self.prepare_data()
    
    def prepare_data(self):
        """Prepare data structures for CP solver."""
        # Get all courses
        self.courses = self.courses_df['Course_ID'].tolist()
        
        # Build course-division pairs (ONE division per course)
        self.divisions = []
        for _, course_row in self.courses_df.iterrows():
            course_id = course_row['Course_ID']
            year = course_row['Year']
            major = course_row['Major']
            dept = course_row['Department']
            
            matching_divs = self.divisions_df[
                (self.divisions_df['Year'] == year) &
                (self.divisions_df['Major'] == major) &
                (self.divisions_df['Department'] == dept)
            ]
            
            if not matching_divs.empty:
                # Pick the first matching division (largest student group)
                best_div = matching_divs.sort_values('StudentNum', ascending=False).iloc[0]
                self.divisions.append((course_id, best_div['Num_ID']))
            else:
                # No matching division found - try matching by Year and Department only
                fallback_divs = self.divisions_df[
                    (self.divisions_df['Year'] == year) &
                    (self.divisions_df['Department'] == dept)
                ]
                if not fallback_divs.empty:
                    best_div = fallback_divs.sort_values('StudentNum', ascending=False).iloc[0]
                    self.divisions.append((course_id, best_div['Num_ID']))
                else:
                    print(f"Warning: No matching division for course {course_id} (Year={year}, Major={major}, Dept={dept})")
        
        # Get available days
        if len(self.doctors_df) > 0 and 'Day' in self.doctors_df.columns:
            self.days = self.doctors_df['Day'].unique().tolist()
        else:
            self.days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Saturday']
        
        self.all_rooms = self.rooms_df['Room_ID'].tolist()
        self.lecture_rooms = self.rooms_df[self.rooms_df['Type'] == 'Lecture']['Room_ID'].tolist()
        self.lab_rooms = self.rooms_df[self.rooms_df['Type'] == 'Lab']['Room_ID'].tolist()
        
        # Define time slots (in minutes since midnight)
        self.time_slots = list(range(8 * 60, 17 * 60 + 1, 60))  # 8 AM to 5 PM, hourly
        
        print(f"Course-division pairs: {len(self.divisions)}")
        print(f"Days: {self.days}")
        print(f"Time slots: {len(self.time_slots)} slots")
    
    def build_model(self):
        """Build the CP-SAT model with all constraints."""
        model = cp_model.CpModel()
        
        # Create decision variables
        # For each course-division pair, we need 'days' number of assignments
        # Each assignment needs: day, room, start_time
        
        # Index mappings
        self.day_indices = {day: i for i, day in enumerate(self.days)}
        self.room_indices = {room: i for i, room in enumerate(self.all_rooms)}
        self.time_indices = {time: i for i, time in enumerate(self.time_slots)}
        
        # Create variables for each assignment
        # assignment_vars[pair_idx][day_idx] = (room_idx, time_idx)
        self.assignment_vars = {}
        
        # For each course-division pair
        for pair_idx, (course_id, div_id) in enumerate(self.divisions):
            course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
            days_needed = int(course_info['Days'])
            
            # Create variables for each required day
            self.assignment_vars[pair_idx] = {}
            
            for day_idx in range(days_needed):
                # Day variable
                day_var = model.NewIntVar(0, len(self.days) - 1, f"day_{pair_idx}_{day_idx}")
                
                # Room variable
                room_var = model.NewIntVar(0, len(self.all_rooms) - 1, f"room_{pair_idx}_{day_idx}")
                
                # Time variable
                time_var = model.NewIntVar(0, len(self.time_slots) - 1, f"time_{pair_idx}_{day_idx}")
                
                self.assignment_vars[pair_idx][day_idx] = {
                    'day': day_var,
                    'room': room_var,
                    'time': time_var,
                    'course_id': course_id,
                    'div_id': div_id
                }
        
        # Add constraints
        self._add_constraints(model)
        
        return model
    
    def _add_constraints(self, model):
        """Add all constraints to the model."""
        
        # For each assignment, get course and division info
        for pair_idx, assignments in self.assignment_vars.items():
            course_id, div_id = self.divisions[pair_idx]
            course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
            div_info = self.divisions_df[self.divisions_df['Num_ID'] == div_id].iloc[0]
            
            instructor_id = course_info['Instructor_ID']
            course_type = course_info['Type']
            hours_per_day = int(course_info['Hours_per_day'])
            duration_minutes = hours_per_day * 60
            student_count = int(div_info['StudentNum'])
            # Hybrid Rotation Logic: Only 50% of students need a room
            student_count = (student_count + 1) // 2
            
            # Get appropriate rooms based on type
            if course_type == 'Lecture':
                suitable_rooms = [self.room_indices[r] for r in self.lecture_rooms 
                                if self.rooms_df[self.rooms_df['Room_ID'] == r]['Capacity'].iloc[0] >= student_count]
                if not suitable_rooms:
                    suitable_rooms = [self.room_indices[r] for r in self.lecture_rooms]
            else:
                suitable_rooms = [self.room_indices[r] for r in self.lab_rooms 
                                if self.rooms_df[self.rooms_df['Room_ID'] == r]['Capacity'].iloc[0] >= student_count]
                if not suitable_rooms:
                    suitable_rooms = [self.room_indices[r] for r in self.lab_rooms]
            
            # Get available days for instructor
            available_day_indices = []
            if instructor_id in doctor_availability:
                for day in doctor_availability[instructor_id].keys():
                    if day in self.day_indices:
                        available_day_indices.append(self.day_indices[day])
            
            if not available_day_indices:
                available_day_indices = list(range(len(self.days)))
            
            # Constraint: Each assignment must use suitable room
            for day_idx, assignment in assignments.items():
                # Room must be suitable for course type and capacity
                model.AddAllowedAssignments([assignment['room']], [(r,) for r in suitable_rooms])
                
                # Day must be available for instructor
                model.AddAllowedAssignments([assignment['day']], [(d,) for d in available_day_indices])
                
                # Time must be valid (not too late for duration)
                # Ensure start_time + duration <= end of day (17:00 = 1020 minutes)
                max_time_idx = len(self.time_slots) - 1
                for time_idx, start_time in enumerate(self.time_slots):
                    end_time = start_time + duration_minutes
                    if end_time > 17 * 60:  # After 5 PM
                        model.Add(assignment['time'] != time_idx)
        
        # Constraint: No conflicts
        # For each pair of assignments, if they conflict, add constraint
        for pair_idx1, assignments1 in self.assignment_vars.items():
            course_id1, div_id1 = self.divisions[pair_idx1]
            course_info1 = self.courses_df[self.courses_df['Course_ID'] == course_id1].iloc[0]
            instructor_id1 = course_info1['Instructor_ID']
            hours_per_day1 = int(course_info1['Hours_per_day'])
            duration1 = hours_per_day1 * 60
            
            for day_idx1, assignment1 in assignments1.items():
                for pair_idx2, assignments2 in self.assignment_vars.items():
                    if pair_idx1 >= pair_idx2:
                        continue
                    
                    course_id2, div_id2 = self.divisions[pair_idx2]
                    course_info2 = self.courses_df[self.courses_df['Course_ID'] == course_id2].iloc[0]
                    instructor_id2 = course_info2['Instructor_ID']
                    hours_per_day2 = int(course_info2['Hours_per_day'])
                    duration2 = hours_per_day2 * 60
                    
                    for day_idx2, assignment2 in assignments2.items():
                        # Room conflict: same room, same day, overlapping times
                        room_conflict = model.NewBoolVar(f"room_conflict_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.Add(assignment1['room'] == assignment2['room']).OnlyEnforceIf(room_conflict)
                        model.Add(assignment1['room'] != assignment2['room']).OnlyEnforceIf(room_conflict.Not())
                        
                        same_day = model.NewBoolVar(f"same_day_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.Add(assignment1['day'] == assignment2['day']).OnlyEnforceIf(same_day)
                        model.Add(assignment1['day'] != assignment2['day']).OnlyEnforceIf(same_day.Not())
                        
                        # Time overlap constraint
                        # We need to check if time ranges overlap
                        # This is complex, so we'll use a simpler approach:
                        # If same room and same day, times must be different
                        # For more accuracy, we'd need to check actual time ranges
                        
                        # Simplified: if same room and same day, enforce time separation
                        both_true = model.NewBoolVar(f"both_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.AddBoolAnd([room_conflict, same_day]).OnlyEnforceIf(both_true)
                        model.AddBoolOr([room_conflict.Not(), same_day.Not()]).OnlyEnforceIf(both_true.Not())
                        
                        # If both room and day are same, times must not overlap
                        # For simplicity, require minimum gap
                        max_duration = max(duration1, duration2) // 60  # Convert to hours
                        time_diff = model.NewIntVar(-len(self.time_slots), len(self.time_slots), 
                                                   f"time_diff_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                        model.Add(time_diff == assignment1['time'] - assignment2['time'])
                        
                        # If both_true, then abs(time_diff) >= max_duration
                        model.AddAbsEquality(time_diff, model.NewIntVar(0, len(self.time_slots), 
                                                                      f"abs_diff_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}"))
                        
                        # Instructor conflict
                        if instructor_id1 == instructor_id2:
                            inst_conflict = model.NewBoolVar(f"inst_conflict_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")
                            model.AddBoolAnd([same_day, 
                                            model.NewBoolVar(f"time_overlap_{pair_idx1}_{day_idx1}_{pair_idx2}_{day_idx2}")])
                            # Simplified: if same instructor and same day, different times
                            model.Add(assignment1['time'] != assignment2['time']).OnlyEnforceIf(same_day)
                        
                        # Division conflict
                        if div_id1 == div_id2:
                            # Same division cannot have overlapping classes
                            model.Add(assignment1['time'] != assignment2['time']).OnlyEnforceIf(same_day)
        
        # Constraint: All days for a course must be different
        for pair_idx, assignments in self.assignment_vars.items():
            if len(assignments) > 1:
                day_vars = [a['day'] for a in assignments.values()]
                # All days must be different
                model.AddAllDifferent(day_vars)
    
    def solve(self):
        """Solve the constraint programming problem."""
        print("Building CP model...")
        model = self.build_model()
        
        print("Creating solver...")
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = self.time_limit_seconds
        
        # Add callback for solution tracking
        solution_collector = SolutionCollector(self.assignment_vars, self.divisions, 
                                             self.days, self.all_rooms, self.time_slots,
                                             self.courses_df, self.divisions_df)
        
        print(f"Solving (time limit: {self.time_limit_seconds}s)...")
        status = solver.Solve(model, solution_collector)
        
        if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
            print(f"Solution found! Status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
            return solution_collector.get_best_solution()
        else:
            print(f"No solution found. Status: {status}")
            return None


class SolutionCollector(cp_model.CpSolverSolutionCallback):
    """Collect solutions during search."""
    
    def __init__(self, assignment_vars, divisions, days, rooms, time_slots, 
                 courses_df, divisions_df):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.assignment_vars = assignment_vars
        self.divisions = divisions
        self.days = days
        self.rooms = rooms
        self.time_slots = time_slots
        self.courses_df = courses_df
        self.divisions_df = divisions_df
        self.solutions = []
    
    def on_solution_callback(self):
        """Called when a solution is found."""
        solution = []
        
        for pair_idx, assignments in self.assignment_vars.items():
            course_id, div_id = self.divisions[pair_idx]
            
            for day_idx, assignment in assignments.items():
                day = self.days[self.Value(assignment['day'])]
                room = self.rooms[self.Value(assignment['room'])]
                start_time = self.time_slots[self.Value(assignment['time'])]
                
                course_info = self.courses_df[self.courses_df['Course_ID'] == course_id].iloc[0]
                hours_per_day = int(course_info['Hours_per_day'])
                end_time = start_time + (hours_per_day * 60)
                
                solution.append({
                    'Day': day,
                    'Course_ID': course_id,
                    'Instructor_ID': course_info['Instructor_ID'],
                    'Group_ID': div_id,
                    'Room_ID': room,
                    'Time_Slot': f"{day}_{start_time}_{end_time}",
                    'Start_Time': start_time,
                    'End_Time': end_time,
                    'Duration': hours_per_day * 60
                })
        
        self.solutions.append(solution)
    
    def get_best_solution(self):
        """Return the best solution found."""
        if self.solutions:
            return self.solutions[-1]  # Return last (best) solution
        return None


print("Constraint Programming scheduler class defined")

In [ ]:
# Create and run CP scheduler
cp_scheduler = SchedulingCP(
    courses_df=Courses,
    rooms_df=Rooms,
    doctors_df=Doctors,
    divisions_df=Divisions,
    time_limit_seconds=300,  # 5 minutes
    use_optimization=True
)

best_schedule = cp_scheduler.solve()

In [ ]:
if best_schedule:
    schedule_data = []
    for assignment in best_schedule:
        course_id = assignment['Course_ID']
        instructor_id = assignment['Instructor_ID']
        group_id = assignment['Group_ID']
        room_id = assignment['Room_ID']
        day = assignment['Day']
        
        course_info = Courses[Courses['Course_ID'] == course_id].iloc[0]
        instructor_info = Doctors[Doctors['Instructor_ID'] == instructor_id].iloc[0]
        division_info = Divisions[Divisions['Num_ID'] == group_id].iloc[0]
        room_info = Rooms[Rooms['Room_ID'] == room_id].iloc[0]
        
        start_time_min = assignment.get('Start_Time', 0)
        end_time_min = assignment.get('End_Time', 0)
        
        start_time_str = minutes_to_time_str(start_time_min)
        end_time_str = minutes_to_time_str(end_time_min)
        
        schedule_data.append({
            'Day': day,
            'Course_Name': course_info['Course_Name'],
            'Instructor_Name': instructor_info['Instructor_Name'],
            'Students': division_info['StudentNum'],
            'Room': room_info['Room'],
            'Start_Time': start_time_str,
            'End_Time': end_time_str,
            'Department': course_info['Department'],
            'Major': course_info['Major']
        })

    Schedule = pd.DataFrame(schedule_data)
    
    print(f"\nGenerated schedule with {len(Schedule)} assignments:")
    print(Schedule.head(20))
    print(f"\nTotal assignments: {len(Schedule)}")
else:
    print("No solution found. The problem may be infeasible or time limit was too short.")

In [ ]:
if best_schedule:
    processed_data_dir = "../../data/Processed Data"
    os.makedirs(processed_data_dir, exist_ok=True)

    output_path = "../../data/Processed Data/Schedule_Output_CP.xlsx"
    Schedule.to_excel(output_path, sheet_name='Schedule', index=False)
    print(f"Schedule saved to {output_path} successfully!")